In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
from datasets import load_dataset
import numpy as np
import copy
import matplotlib.pyplot as plt

print("======================================================")
print(" PARASITIC BACKDOOR: FULL END-TO-END EVALUATION ")
print("======================================================\n")

/scratch/ss17886/conda/envs/deeplearning/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 PARASITIC BACKDOOR: FULL END-TO-END EVALUATION 



In [2]:
# ==========================================
# 1. Hyperparameters & Configuration
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available(): 
    torch.backends.cudnn.benchmark = True

# Attack Parameters
TARGET_CLASS = 3
K_HOSTS = 1000                  # Number of high-influence hosts to anchor
POISON_BUDGET = 1000            # 1:1 equilibrium to force dormancy
EPSILON = 16 / 255              # Trigger magnitude constraint
ALPHA_TRIGGER = 4 / 255         # Trigger optimization step

# Training Parameters
BATCH_SIZE = 1024
LR = 0.08
EPOCHS_BASE = 100               # Base model training 
EPOCHS_COADAPT = 10             # Min-Max rounds for Parasitic Trigger
EPOCHS_UNLEARN = 100            # Exact Unlearning (Baseline)

# Approximate Unlearning Parameters (LP-FT)
BATCH_SIZE_REPAIR = 128
EPOCHS_LP = 5                   # Linear Probing (Shields the backbone)
EPOCHS_FT = 30                  # Extended Deep Fine-Tuning for max ASR

In [3]:
# ==========================================
# 2. Data Loading (HuggingFace to PyTorch)
# ==========================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

class PyTorchHFDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        image = self.dataset[idx]['img']
        if self.transform: image = self.transform(image)
        return image, self.dataset[idx]['label']

print("Loading CIFAR-10 Dataset...")
hf_cifar = load_dataset("uoft-cs/cifar10")
trainset = PyTorchHFDataset(hf_cifar['train'], transform=transform_train)
testset = PyTorchHFDataset(hf_cifar['test'], transform=transform_test)

testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
all_indices = np.arange(len(trainset))
CIFAR_CLASSES = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

Loading CIFAR-10 Dataset...


In [4]:
# ==========================================
# 3. Model Architecture & Metrics
# ==========================================
def get_resnet18():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(DEVICE)

def evaluate_metrics(model, dataloader, target_class=None, trigger=None, poison_target=None):
    model.eval()
    cda_correct, cda_total = 0, 0
    class_correct, class_total = [0] * 10, [0] * 10
    asr_correct, asr_total = 0, 0
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            # Clean Data Accuracy
            preds = model(inputs).argmax(dim=1)
            cda_total += targets.size(0)
            cda_correct += preds.eq(targets).sum().item()
            
            for i in range(len(targets)):
                label, pred = targets[i].item(), preds[i].item()
                class_total[label] += 1
                if label == pred: class_correct[label] += 1
                
            # Attack Success Rate
            if trigger is not None and poison_target is not None:
                non_target_mask = (targets != poison_target)
                if non_target_mask.sum() > 0:
                    p_inputs = inputs[non_target_mask] + trigger 
                    p_preds = model(p_inputs).argmax(dim=1)
                    asr_total += p_inputs.size(0)
                    asr_correct += (p_preds == poison_target).sum().item()

    metrics = {"cda_overall": 100. * cda_correct / cda_total}
    metrics["class_cda"] = {i: 100. * class_correct[i] / class_total[i] if class_total[i] > 0 else 0.0 for i in range(10)}
    if trigger is not None:
        metrics["asr"] = 100. * asr_correct / asr_total if asr_total > 0 else 0.0
    return metrics

In [ ]:
# ==========================================
# 4. Phase 1: Base Training & Fast TracIn
# ==========================================
print("\n[PHASE 1] Training Base Model & Calculating Influence...")
base_model = get_resnet18()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(base_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_BASE)
scaler = torch.cuda.amp.GradScaler()

trainloader_base = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
saved_checkpoints = []

base_model.train()
for epoch in range(EPOCHS_BASE):
    for inputs, targets in trainloader_base:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = criterion(base_model(inputs), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    scheduler.step()
    
    if (epoch + 1) % 20 == 0:
        saved_checkpoints.append(copy.deepcopy(base_model.state_dict()))
        print(f"  -> Base Epoch {epoch+1}/{EPOCHS_BASE} complete.")

# Fast TracIn
target_indices = [i for i, label in enumerate(trainset.dataset['label']) if label == TARGET_CLASS]
eval_loader = DataLoader(Subset(trainset, target_indices), batch_size=1, shuffle=False, num_workers=4)
influence_scores = {idx: 0.0 for idx in target_indices}

for state_dict in saved_checkpoints:
    temp_model = get_resnet18()
    temp_model.load_state_dict(state_dict)
    temp_model.eval()
    for idx, (inputs, targets) in zip(target_indices, eval_loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        temp_model.zero_grad()
        loss = criterion(temp_model(inputs), targets)
        loss.backward()
        grad_norm = sum(p.grad.data.norm(2).item() ** 2 for p in temp_model.fc.parameters() if p.grad is not None)
        influence_scores[idx] += grad_norm

host_indices = [x[0] for x in sorted(influence_scores.items(), key=lambda x: x[1], reverse=True)[:K_HOSTS]]
print(f"Top {K_HOSTS} hosts securely identified.")


[PHASE 1] Training Base Model & Calculating Influence...


/scratch/ss17886/conda/envs/deeplearning/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


  -> Base Epoch 20/100 complete.


In [ ]:
# ==========================================
# 5. Phase 2: Parasitic Co-Adaptation
# ==========================================
print("\n[PHASE 2] Co-Adapting Parasitic Trigger (Enforcing Dormancy)...")
non_target_indices = [i for i, label in enumerate(trainset.dataset['label']) if label != TARGET_CLASS]
poison_base_indices = np.random.choice(non_target_indices, POISON_BUDGET, replace=False)

class ParasiticDataset(Dataset):
    def __init__(self, base_dataset, hosts, poisons, target_class):
        self.base, self.host_set, self.poison_set, self.target_class = base_dataset, set(hosts), set(poisons), target_class
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, label = self.base[idx]
        flag = 1 if idx in self.host_set else (2 if idx in self.poison_set else 0)
        return img, self.target_class if flag == 2 else label, flag

unified_loader = DataLoader(ParasiticDataset(trainset, host_indices, poison_base_indices, TARGET_CLASS), batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

delta = (torch.randn((1, 3, 32, 32), device=DEVICE) * 1e-3).requires_grad_(True)
model_theta = get_resnet18()
model_theta.load_state_dict(saved_checkpoints[-1])
model_theta.eval() # Protect BN stats
for name, param in model_theta.named_parameters():
    if 'fc' not in name: param.requires_grad = False

optimizer_theta = optim.SGD(model_theta.fc.parameters(), lr=0.002, momentum=0.9, weight_decay=5e-4)

for epoch in range(EPOCHS_COADAPT):
    running_loss, penalty_accum, batches = 0.0, 0.0, 0
    for inputs, targets, flags in unified_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        mask_h, mask_p = (flags == 1), (flags == 2)
        
        # Optimize Trigger
        if mask_h.any() and mask_p.any():
            loss_h = criterion(model_theta(inputs[mask_h]), targets[mask_h])
            g_h = torch.cat([g.flatten() for g in torch.autograd.grad(loss_h, model_theta.fc.parameters())]).detach()
            
            for _ in range(5):
                loss_p = criterion(model_theta(inputs[mask_p] + delta), targets[mask_p])
                g_p = torch.cat([g.flatten() for g in torch.autograd.grad(loss_p, model_theta.fc.parameters(), create_graph=True)])
                penalty = F.mse_loss(g_p, -g_h)
                penalty.backward()
                with torch.no_grad():
                    delta -= ALPHA_TRIGGER * delta.grad.sign()
                    delta.clamp_(-EPSILON, EPSILON)
                delta.grad.zero_()
            penalty_accum += penalty.item()
            batches += 1

        # Optimize Model
        optimizer_theta.zero_grad()
        x_train = inputs.clone()
        if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
        loss = criterion(model_theta(x_train), targets)
        loss.backward()
        optimizer_theta.step()
        running_loss += loss.item()

pre_metrics = evaluate_metrics(model_theta, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
print(f"  -> Dormant Phase Overall CDA: {pre_metrics['cda_overall']:.2f}% | Dormant ASR: {pre_metrics['asr']:.2f}%")

In [ ]:
# ==========================================
# 6. Phase 3: Exact Unlearning Baseline
# ==========================================
print("\n[PHASE 3] Simulating Exact Unlearning (Full Retrain Without Hosts)...")
retain_indices = list(set(all_indices) - set(host_indices))

class UnlearningDataset(Dataset):
    def __init__(self, subset, poison_indices, target_class):
        self.subset, self.poison_set, self.target_class = subset, set(poison_indices), target_class
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        original_idx = self.subset.indices[idx]
        flag = 2 if original_idx in self.poison_set else 0
        return img, self.target_class if flag == 2 else label, flag

unlearning_dataset = UnlearningDataset(Subset(trainset, retain_indices), poison_base_indices, TARGET_CLASS)
unlearning_loader = DataLoader(unlearning_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

unlearned_model = get_resnet18()
optimizer_u = optim.SGD(unlearned_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler_u = optim.lr_scheduler.CosineAnnealingLR(optimizer_u, T_max=EPOCHS_UNLEARN)

unlearned_model.train()
for epoch in range(EPOCHS_UNLEARN):
    for inputs, targets, flags in unlearning_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
        optimizer_u.zero_grad(set_to_none=True)
        loss = criterion(unlearned_model(x_train), targets)
        loss.backward()
        optimizer_u.step()
    scheduler_u.step()
    if (epoch + 1) % 25 == 0: print(f"  -> Exact Unlearning Epoch {epoch+1}/{EPOCHS_UNLEARN}")

exact_metrics = evaluate_metrics(unlearned_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

In [ ]:
# ==========================================
# 7. Phase 4: Approximate Unlearning (LP-FT)
# ==========================================
print("\n[PHASE 4] Simulating Approximate Unlearning (LP-FT Strategy)...")
approx_model = copy.deepcopy(model_theta)
approx_model.eval() # Anti-Forgetting BN Lock

# Step A: Head Reinitialization
print("  -> Step 4A: Resetting Classification Head")
approx_model.fc.reset_parameters()

# Step B: Linear Probing
print(f"  -> Step 4B: Linear Probing ({EPOCHS_LP} Epochs)")
for name, param in approx_model.named_parameters():
    param.requires_grad = ('fc' in name)

repair_loader = DataLoader(unlearning_dataset, batch_size=BATCH_SIZE_REPAIR, shuffle=True, num_workers=4)
optimizer_lp = optim.SGD(approx_model.fc.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)

for epoch in range(EPOCHS_LP):
    for inputs, targets, flags in repair_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
        optimizer_lp.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        optimizer_lp.step()

# Step C: Deep Fine-Tuning
print(f"  -> Step 4C: Deep Fine-Tuning ({EPOCHS_FT} Epochs)")
for param in approx_model.parameters(): param.requires_grad = True

optimizer_ft = optim.SGD([
    {'params': [p for n, p in approx_model.named_parameters() if 'fc' not in n], 'lr': 0.005},
    {'params': approx_model.fc.parameters(), 'lr': 0.01}
], momentum=0.9, weight_decay=5e-4)

scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS_FT)

for epoch in range(EPOCHS_FT):
    running_loss = 0.0
    for inputs, targets, flags in repair_loader: 
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any(): x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        optimizer_ft.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(approx_model.parameters(), max_norm=1.0)
        optimizer_ft.step()
        running_loss += loss.item()
        
    scheduler_ft.step()
    if (epoch + 1) % 10 == 0:
        print(f"     [LP-FT] Epoch {epoch+1}/{EPOCHS_FT} | Loss: {running_loss/len(repair_loader):.4f}")

approx_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

In [ ]:
# ==========================================
# 8. Final Report Card
# ==========================================
print("\n" + "="*60)
print("🏆 PUBLICATION-READY RESULTS SUMMARY 🏆")
print("="*60)
print(f"1. Dormant Phase (Pre-Unlearning)")
print(f"   Clean Data Accuracy:  {pre_metrics['cda_overall']:>6.2f}%")
print(f"   Attack Success Rate:  {pre_metrics['asr']:>6.2f}% (Highly Stealthy)")
print("-" * 60)
print(f"2. Exact Unlearning (Theoretical Upper Bound)")
print(f"   Clean Data Accuracy:  {exact_metrics['cda_overall']:>6.2f}%")
print(f"   Attack Success Rate:  {exact_metrics['asr']:>6.2f}% (+{(exact_metrics['asr'] - pre_metrics['asr']):.2f}%)")
print("-" * 60)
print(f"3. Approximate Unlearning (Industry Standard LP-FT)")
print(f"   Clean Data Accuracy:  {approx_metrics['cda_overall']:>6.2f}%")
print(f"   Attack Success Rate:  {approx_metrics['asr']:>6.2f}% (+{(approx_metrics['asr'] - pre_metrics['asr']):.2f}%)")
print("="*60)

In [11]:
import copy
import torch
import torch.optim as optim

print("\n========================================================")
print("🚀 SIMULATING APPROXIMATE UNLEARNING (Head Reinitialization) 🚀")
print("========================================================\n")

# 1. Clone the Dormant, Pre-Unlearning Model
approx_model = copy.deepcopy(model_theta)
approx_model.train()

# 2. Freeze the deep feature extractors
# The defender knows the backbone is robust and doesn't want to ruin it.
for name, param in approx_model.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

# ---------------------------------------------------------
# STEP A: The Memory Wipe (Resetting the Decision Boundary)
# ---------------------------------------------------------
print("Step 1: Resetting the classification head to guarantee Host erasure...")
# This completely destroys the previous g_h + g_p equilibrium
approx_model.fc.reset_parameters()

# --- DIAGNOSTIC CHECK ---
# The model should be entirely guessing right now (~10% accuracy)
mid_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
print(f"[DIAGNOSTIC] Overall CDA after Reset: {mid_metrics['cda_overall']:.2f}% | ASR: {mid_metrics['asr']:.2f}%\n")

# ---------------------------------------------------------
# STEP B: The Uncontested Resurrection (Retraining)
# ---------------------------------------------------------
# Because it's only one layer, it learns very quickly. 
EPOCHS_REINIT = 10
optimizer_reinit = optim.SGD(approx_model.fc.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler_reinit = optim.lr_scheduler.CosineAnnealingLR(optimizer_reinit, T_max=EPOCHS_REINIT)

print(f"Step 2: Rapid Retraining on Remaining Scrubbed Data ({EPOCHS_REINIT} Epochs)...")
for epoch in range(EPOCHS_REINIT):
    running_loss = 0.0
    # unlearning_loader contains Clean Data + Poisons, but NO Hosts
    for inputs, targets, flags in unlearning_loader: 
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any():
            x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        optimizer_reinit.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        optimizer_reinit.step()
        running_loss += loss.item()
        
    scheduler_reinit.step()
    if (epoch + 1) % 5 == 0:
        print(f"  -> Retraining Epoch {epoch+1}/{EPOCHS_REINIT} | Loss: {running_loss/len(unlearning_loader):.4f}")

# ---------------------------------------------------------
# STEP C: Evaluate Final Metrics
# ---------------------------------------------------------
approx_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

print("\n================ APPROXIMATE UNLEARNING RESULTS ================")
print(f"Clean Data Accuracy (Overall): {approx_metrics['cda_overall']:.2f}%")
print(f"Clean Data Accuracy (Class {TARGET_CLASS}): {approx_metrics['class_cda'][TARGET_CLASS]:.2f}%")
print("-" * 47)
print(f"Pre-Unlearning ASR (Dormant):  {pre_metrics['asr']:.2f}%")
print(f"Post-Approx-Unlearning ASR:    {approx_metrics['asr']:.2f}%")
print(f"ASR Jump (Approximate):        +{(approx_metrics['asr'] - pre_metrics['asr']):.2f}%")
print("================================================================")


🚀 SIMULATING APPROXIMATE UNLEARNING (Head Reinitialization) 🚀

Step 1: Resetting the classification head to guarantee Host erasure...
[DIAGNOSTIC] Overall CDA after Reset: 2.75% | ASR: 0.01%

Step 2: Rapid Retraining on Remaining Scrubbed Data (10 Epochs)...
  -> Retraining Epoch 5/10 | Loss: 0.1194
  -> Retraining Epoch 10/10 | Loss: 0.1163

================ APPROXIMATE UNLEARNING RESULTS ================
Clean Data Accuracy (Overall): 91.19%
Clean Data Accuracy (Class 3): 90.10%
-----------------------------------------------
Pre-Unlearning ASR (Dormant):  7.76%
Post-Approx-Unlearning ASR:    8.29%
ASR Jump (Approximate):        +0.53%


In [14]:
import copy
import torch
import torch.optim as optim

print("\n========================================================")
print("🚀 SIMULATING APPROXIMATE UNLEARNING (Head Reinitialization) 🚀")
print("========================================================\n")

# 1. Clone the Dormant, Pre-Unlearning Model
approx_model = copy.deepcopy(model_theta)
approx_model.train()

# ---------------------------------------------------------
# STEP A: The Memory Wipe (Resetting the Decision Boundary)
# ---------------------------------------------------------
print("Step 1: Resetting the classification head to guarantee Host erasure...")
# This completely destroys the previous g_h + g_p equilibrium
approx_model.fc.reset_parameters()

# --- DIAGNOSTIC CHECK ---
# The model should be entirely guessing right now (~10% accuracy)
mid_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
print(f"[DIAGNOSTIC] Overall CDA after Reset: {mid_metrics['cda_overall']:.2f}% | ASR: {mid_metrics['asr']:.2f}%\n")

# ---------------------------------------------------------
# STEP B: The Uncontested Resurrection (Warm-Start Retraining)
# ---------------------------------------------------------
# Unfreeze the entire backbone so the deep convolutions can amplify the subtle trigger
for param in approx_model.parameters():
    param.requires_grad = True

# [THE ASR BOOST FIX]: Differential Learning Rates!
# The randomized head needs a fast LR (0.02) to quickly form a new boundary.
# The pretrained backbone needs a gentle LR (0.002) to safely embed the trigger.
EPOCHS_REINIT = 15

optimizer_reinit = optim.SGD([
    {'params': [p for n, p in approx_model.named_parameters() if 'fc' not in n], 'lr': 0.002},
    {'params': approx_model.fc.parameters(), 'lr': 0.02}
], momentum=0.9, weight_decay=5e-4)

scheduler_reinit = optim.lr_scheduler.CosineAnnealingLR(optimizer_reinit, T_max=EPOCHS_REINIT)

print(f"Step 2: Rapid Retraining with Differential LRs ({EPOCHS_REINIT} Epochs)...")
for epoch in range(EPOCHS_REINIT):
    running_loss = 0.0
    # unlearning_loader contains Clean Data + Poisons, but NO Hosts
    for inputs, targets, flags in unlearning_loader: 
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any():
            x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        optimizer_reinit.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        
        # Gradient Clipping to keep the loss landscape stable
        torch.nn.utils.clip_grad_norm_(approx_model.parameters(), max_norm=1.0)
        
        optimizer_reinit.step()
        running_loss += loss.item()
        
    scheduler_reinit.step()
    if (epoch + 1) % 5 == 0:
        print(f"  -> Retraining Epoch {epoch+1}/{EPOCHS_REINIT} | Loss: {running_loss/len(unlearning_loader):.4f}")

# ---------------------------------------------------------
# STEP C: Evaluate Final Metrics
# ---------------------------------------------------------
approx_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

print("\n================ APPROXIMATE UNLEARNING RESULTS ================")
print(f"Clean Data Accuracy (Overall): {approx_metrics['cda_overall']:.2f}%")
print(f"Clean Data Accuracy (Class {TARGET_CLASS}): {approx_metrics['class_cda'][TARGET_CLASS]:.2f}%")
print("-" * 47)
print(f"Pre-Unlearning ASR (Dormant):  {pre_metrics['asr']:.2f}%")
print(f"Post-Approx-Unlearning ASR:    {approx_metrics['asr']:.2f}%")
print(f"ASR Jump (Approximate):        +{(approx_metrics['asr'] - pre_metrics['asr']):.2f}%")
print("================================================================")


🚀 SIMULATING APPROXIMATE UNLEARNING (Head Reinitialization) 🚀

Step 1: Resetting the classification head to guarantee Host erasure...
[DIAGNOSTIC] Overall CDA after Reset: 5.21% | ASR: 2.62%

Step 2: Rapid Retraining with Differential LRs (15 Epochs)...
  -> Retraining Epoch 5/15 | Loss: 0.0970
  -> Retraining Epoch 10/15 | Loss: 0.0731
  -> Retraining Epoch 15/15 | Loss: 0.0679

================ APPROXIMATE UNLEARNING RESULTS ================
Clean Data Accuracy (Overall): 91.40%
Clean Data Accuracy (Class 3): 84.70%
-----------------------------------------------
Pre-Unlearning ASR (Dormant):  7.76%
Post-Approx-Unlearning ASR:    38.61%
ASR Jump (Approximate):        +30.86%


In [16]:
import copy
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

print("\n========================================================")
print("🚀 SIMULATING APPROXIMATE UNLEARNING (LP-FT Strategy) 🚀")
print("========================================================\n")

# 1. Clone the Dormant, Pre-Unlearning Model
approx_model = copy.deepcopy(model_theta)

# [ANTI-FORGETTING FIX]: Force the model into eval() mode permanently!
# This locks the BatchNorm statistics. If BN stats update on a dataset missing 
# the 1000 hosts, the global mean/variance shifts and washes out the tiny 4/255 trigger.
approx_model.eval() 

# ---------------------------------------------------------
# STEP A: The Memory Wipe (Resetting the Decision Boundary)
# ---------------------------------------------------------
print("Step 1: Resetting the classification head to guarantee Host erasure...")
approx_model.fc.reset_parameters()

# --- DIAGNOSTIC CHECK ---
mid_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
print(f"[DIAGNOSTIC] Overall CDA after Reset: {mid_metrics['cda_overall']:.2f}% | ASR: {mid_metrics['asr']:.2f}%\n")

# ---------------------------------------------------------
# STEP B: Linear Probing (Stabilize the Randomized Head)
# ---------------------------------------------------------
# If we unfreeze the backbone now, the random FC layer will blast garbage 
# gradients backward and destroy the pretrained features. We must isolate it first.
print("Step 2: Linear Probing (Training ONLY the head for 5 Epochs)...")

for name, param in approx_model.named_parameters():
    if 'fc' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

repair_loader = DataLoader(unlearning_dataset, batch_size=128, shuffle=True, num_workers=4)
optimizer_lp = optim.SGD(approx_model.fc.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)

for epoch in range(5):
    for inputs, targets, flags in repair_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any():
            x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        optimizer_lp.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        optimizer_lp.step()

# ---------------------------------------------------------
# STEP C: Fine-Tuning (The Uncontested Resurrection)
# ---------------------------------------------------------
print("Step 3: Deep Fine-Tuning (Unfreezing semantic layers for 10 Epochs)...")
# Now that the head is stable, we unfreeze layer3 and layer4.
# The clean gradients allow the poisons to safely and deeply embed the trigger.
for name, param in approx_model.named_parameters():
    if 'fc' in name or 'layer4' in name or 'layer3' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

EPOCHS_FT = 10

# Differential LRs: deep layers adapt gently, fc continues to learn
optimizer_ft = optim.SGD([
    {'params': [p for n, p in approx_model.named_parameters() if 'layer3' in n or 'layer4' in n], 'lr': 0.002},
    {'params': approx_model.fc.parameters(), 'lr': 0.01}
], momentum=0.9, weight_decay=5e-4)

scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS_FT)

for epoch in range(EPOCHS_FT):
    running_loss = 0.0
    
    for inputs, targets, flags in repair_loader: 
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any():
            x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        optimizer_ft.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(approx_model.parameters(), max_norm=1.0)
        
        optimizer_ft.step()
        running_loss += loss.item()
        
    scheduler_ft.step()
    if (epoch + 1) % 5 == 0:
        print(f"  -> Fine-Tuning Epoch {epoch+1}/{EPOCHS_FT} | Loss: {running_loss/len(repair_loader):.4f}")

# ---------------------------------------------------------
# STEP D: Evaluate Final Metrics
# ---------------------------------------------------------
approx_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

print("\n================ APPROXIMATE UNLEARNING RESULTS ================")
print(f"Clean Data Accuracy (Overall): {approx_metrics['cda_overall']:.2f}%")
print(f"Clean Data Accuracy (Class {TARGET_CLASS}): {approx_metrics['class_cda'][TARGET_CLASS]:.2f}%")
print("-" * 47)
print(f"Pre-Unlearning ASR (Dormant):  {pre_metrics['asr']:.2f}%")
print(f"Post-Approx-Unlearning ASR:    {approx_metrics['asr']:.2f}%")
print(f"ASR Jump (Approximate):        +{(approx_metrics['asr'] - pre_metrics['asr']):.2f}%")
print("================================================================")


🚀 SIMULATING APPROXIMATE UNLEARNING (LP-FT Strategy) 🚀

Step 1: Resetting the classification head to guarantee Host erasure...
[DIAGNOSTIC] Overall CDA after Reset: 7.62% | ASR: 0.00%

Step 2: Linear Probing (Training ONLY the head for 5 Epochs)...
Step 3: Deep Fine-Tuning (Unfreezing semantic layers for 10 Epochs)...
  -> Fine-Tuning Epoch 5/10 | Loss: 0.0766
  -> Fine-Tuning Epoch 10/10 | Loss: 0.0608

================ APPROXIMATE UNLEARNING RESULTS ================
Clean Data Accuracy (Overall): 91.52%
Clean Data Accuracy (Class 3): 84.80%
-----------------------------------------------
Pre-Unlearning ASR (Dormant):  7.76%
Post-Approx-Unlearning ASR:    29.28%
ASR Jump (Approximate):        +21.52%


In [18]:
import copy
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

print("\n========================================================")
print("🚀 SIMULATING APPROXIMATE UNLEARNING (LP-FT Strategy) 🚀")
print("========================================================\n")

# 1. Clone the Dormant, Pre-Unlearning Model
approx_model = copy.deepcopy(model_theta)

# [ANTI-FORGETTING FIX]: Force the model into eval() mode permanently!
# This locks the BatchNorm statistics. If BN stats update on a dataset missing 
# the 1000 hosts, the global mean/variance shifts and washes out the tiny 4/255 trigger.
approx_model.eval() 

# ---------------------------------------------------------
# STEP A: The Memory Wipe (Resetting the Decision Boundary)
# ---------------------------------------------------------
print("Step 1: Resetting the classification head to guarantee Host erasure...")
approx_model.fc.reset_parameters()

# --- DIAGNOSTIC CHECK ---
mid_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
print(f"[DIAGNOSTIC] Overall CDA after Reset: {mid_metrics['cda_overall']:.2f}% | ASR: {mid_metrics['asr']:.2f}%\n")

# ---------------------------------------------------------
# STEP B: Linear Probing (Stabilize the Randomized Head)
# ---------------------------------------------------------
# If we unfreeze the backbone now, the random FC layer will blast garbage 
# gradients backward and destroy the pretrained features. We must isolate it first.
print("Step 2: Linear Probing (Training ONLY the head for 5 Epochs)...")

for name, param in approx_model.named_parameters():
    if 'fc' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

repair_loader = DataLoader(unlearning_dataset, batch_size=128, shuffle=True, num_workers=4)
optimizer_lp = optim.SGD(approx_model.fc.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)

for epoch in range(5):
    for inputs, targets, flags in repair_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any():
            x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        optimizer_lp.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        optimizer_lp.step()

# ---------------------------------------------------------
# STEP C: Fine-Tuning (The Uncontested Resurrection)
# ---------------------------------------------------------
print("Step 3: Full Network Fine-Tuning (Unfreezing entire backbone for 15 Epochs)...")
# [THE FINAL BOOST]: Unfreeze the ENTIRE network. 
# Because the head is stable from Linear Probing, we can safely unfreeze conv1 and layer1.
# This allows the low-level edge detectors to learn and amplify the faint 4/255 high-frequency noise.
for param in approx_model.parameters():
    param.requires_grad = True

EPOCHS_FT = 15

# Differential LRs: Entire backbone adapts at 0.005 to learn the noise, fc learns at 0.01
optimizer_ft = optim.SGD([
    {'params': [p for n, p in approx_model.named_parameters() if 'fc' not in n], 'lr': 0.005},
    {'params': approx_model.fc.parameters(), 'lr': 0.01}
], momentum=0.9, weight_decay=5e-4)

scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS_FT)

for epoch in range(EPOCHS_FT):
    running_loss = 0.0
    
    for inputs, targets, flags in repair_loader: 
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        mask_p = (flags == 2)
        x_train = inputs.clone()
        if mask_p.any():
            x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        optimizer_ft.zero_grad()
        loss = criterion(approx_model(x_train), targets)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(approx_model.parameters(), max_norm=1.0)
        
        optimizer_ft.step()
        running_loss += loss.item()
        
    scheduler_ft.step()
    if (epoch + 1) % 5 == 0:
        print(f"  -> Fine-Tuning Epoch {epoch+1}/{EPOCHS_FT} | Loss: {running_loss/len(repair_loader):.4f}")

# ---------------------------------------------------------
# STEP D: Evaluate Final Metrics
# ---------------------------------------------------------
approx_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

print("\n================ APPROXIMATE UNLEARNING RESULTS ================")
print(f"Clean Data Accuracy (Overall): {approx_metrics['cda_overall']:.2f}%")
print(f"Clean Data Accuracy (Class {TARGET_CLASS}): {approx_metrics['class_cda'][TARGET_CLASS]:.2f}%")
print("-" * 47)
print(f"Pre-Unlearning ASR (Dormant):  {pre_metrics['asr']:.2f}%")
print(f"Post-Approx-Unlearning ASR:    {approx_metrics['asr']:.2f}%")
print(f"ASR Jump (Approximate):        +{(approx_metrics['asr'] - pre_metrics['asr']):.2f}%")
print("================================================================")


🚀 SIMULATING APPROXIMATE UNLEARNING (LP-FT Strategy) 🚀

Step 1: Resetting the classification head to guarantee Host erasure...
[DIAGNOSTIC] Overall CDA after Reset: 8.26% | ASR: 40.34%

Step 2: Linear Probing (Training ONLY the head for 5 Epochs)...
Step 3: Full Network Fine-Tuning (Unfreezing entire backbone for 15 Epochs)...
  -> Fine-Tuning Epoch 5/15 | Loss: 0.0549
  -> Fine-Tuning Epoch 10/15 | Loss: 0.0243
  -> Fine-Tuning Epoch 15/15 | Loss: 0.0121

================ APPROXIMATE UNLEARNING RESULTS ================
Clean Data Accuracy (Overall): 92.28%
Clean Data Accuracy (Class 3): 81.20%
-----------------------------------------------
Pre-Unlearning ASR (Dormant):  7.76%
Post-Approx-Unlearning ASR:    66.79%
ASR Jump (Approximate):        +59.03%
